In [ ]:
import sys
import os

# Agregar el directorio padre al sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime


import cashia_core.data_channel.promok_datapipe_func as pkdp
import cashia_core.common_tools.pm_mxgeographic as pm_mxgeo

import cashia_mlp.promok_models_config as pkconfig
from cashia_mlp.dataframe_tracker import *


%matplotlib inline

In [ ]:
import sklearn as sk

print("numpy:",np.__version__)
print("pandas:",pd.__version__)
print("seborn:",sns.__version__)
print("sklearn:",sk.__version__)
#print(":",.__version__)

# 1 Procesar los datos

## 1.1 Lectura de los datos

### 1.1.1 Lectura de los datos de configuración

In [ ]:
print('Configuration file: ', pkconfig.configuration_file_name)
print('Configuration : ', pkconfig.conf_column)
all_features = pd.read_excel("./config/" + pkconfig.configuration_file_name, index_col=0)

In [ ]:
features = all_features[all_features[pkconfig.conf_column] == 'Sí']
features

In [ ]:
features_to_use = set(features['Característica'].values)
features_to_use

In [ ]:
agregated_feautures = set(features[features['Sección'] == 'Data Agregation']['Característica'].values)
agregated_feautures

### 1.1.2 Lectura de los datos de entrenamiento

In [ ]:
print('Source file: ', pkconfig.source_file_name)
#data = pd.read_excel(pkconfig.source_file_name)
data = pd.read_csv("./data/" + pkconfig.source_file_name, index_col=0)

data_tracker = DataFrameEvolutionTracker(data)
data_tracker.to_dataframe()

if pkconfig.conf_column == "NV_Agt" or pkconfig.conf_column == "RNV_Agt":
    print("Agent score")
    data = data[data['Score Agt'] != 0]
else:
    print("Using CC score")

data_tracker.register_step(f"First filter for {pkconfig.conf_column}", data)
data_tracker.to_dataframe()

data.head()

# Filtros especiales

## Borramos los datos de créditos no aprobados

In [ ]:
# Quitamos esta línea en diciembre 2025 ya que en teoría sólo me pasan los créditos aprobados
# data = data[data['Aprobado'] == 1]
# data.shape

## De acuerdo a la columna de configuración selecionamos los datos

In [ ]:
# Caso 1: Renovaciones
if pkconfig.conf_column == "NV_Agt":
    data = data[data['Categoria'] == "NV"]
    
elif pkconfig.conf_column == "RNV_Agt":
    data = data[(data['Categoria'] == "RNV")]
    
elif pkconfig.conf_column == "NV_CC":
    data = data[(data['Categoria'] == "NV")]
    
elif pkconfig.conf_column == "RNV_CC":
    data = data[(data['Categoria'] == "RNV")]

elif pkconfig.conf_column == "NV_Agt_CS":
    data = data[data['Categoria'] == "NV"]
    
elif pkconfig.conf_column == "RNV_Agt_CS":
    data = data[(data['Categoria'] == "RNV")]
    
elif pkconfig.conf_column == "NV_CC_CS":
    data = data[(data['Categoria'] == "NV")]
    
elif pkconfig.conf_column == "RNV_CC_CS":
    data = data[(data['Categoria'] == "RNV")]
    
else:
    print("Invalid option")
    exit()

data_tracker.register_step(f"Second filter for {pkconfig.conf_column}", data)
data_tracker.to_dataframe()

## 1.2 Explorar los datos

In [ ]:
data_features = set(data.columns)
data_features

In [ ]:
features_to_drop = list(data_features - features_to_use)
# Removemos de la lista de elementos a borrar 'Fecha Inicial ya que la necesitaremos para los intervalos de aprendizaje'
features_to_drop.remove('Fecha Inicial')
features_to_drop

In [ ]:
features

In [ ]:
int_features_list = features[(features['Sub tipo'] == "Entero") & (features['Sección'] != "Data Agregation")]
int_features_list = int_features_list['Característica'].unique()
int_features_list = list(int_features_list)

In [ ]:
int_features_list

In [ ]:
all_features

In [ ]:
if ('CP' not in features_to_use) and (('CP_Latitud' in features_to_use)| ('CP_Longitud' in features_to_use)):
    int_features_list.append('CP')
    
if ('CP Aval' not in features_to_use) and (('CP_Aval_Latitud' in features_to_use) | ('CP_Aval_Longitud' in features_to_use)):
    int_features_list.append('CP Aval')                                        

if ('Score CC' not in features_to_use) and 'Odds' in features_to_use:
    int_features_list.append('Score CC')

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

## 1.3 Limpiar los datos

In [ ]:
quantitative_features = set(features[(features['Tipo de dato'] == 'Cuantitativo') & (features['Tipo de variable'] == 'Predictor')]['Característica'].values)
quantitative_features

In [ ]:
categorical_features = set(features[(features['Tipo de dato'] == 'Categórico') & (features['Tipo de variable'] == 'Predictor')]['Característica'].values)
categorical_features

### 1.3.1 Limpiar ingresos de conyuge para los que tienen NA poner "Desconocido"

In [ ]:
if 'Ingreso Conyuge' in features_to_use: 
    pkdp.clean_ingreso_conyuge(data)
data_tracker.register_step(f"Cleaning datos conyuge", data)
data_tracker.to_dataframe()

### 1.3.1b

In [ ]:
#pk.clean_incobrable(data)

### 1.3.2 Poner los egresos igual a sus ingresos a aquellos que tienen vacío en egresos

In [ ]:
pkdp.clean_egresos(data)
data_tracker.register_step(f"Cleaning egresos", data)
data_tracker.to_dataframe()

In [ ]:
data.isnull().sum()

In [ ]:
raw_data = features_to_use-agregated_feautures
raw_data

In [ ]:
data[list(raw_data)].isnull().sum()

### 1.3.3. Llenamos con "Unknown" los valores de columnas categóricas vacias

In [ ]:
#data[list(categorical_features-agregated_feautures)] = data[list(categorical_features-agregated_feautures)].fillna('Unknown')

### 1.3.4 Borrar renglones con datos nulos en las columnas que se usarán en el modelo

In [ ]:
data = data.dropna(subset=list(features_to_use-agregated_feautures))
data_tracker.register_step(f"Cleaning for columns with null", data)
data_tracker.to_dataframe()

In [ ]:
data.isnull().sum()

### 1.3.5 Convertir a entero las columnas que deberían ser entero y no lo son

In [ ]:
# 1) Convertir a numérico con NaN donde haya "NC" o vacío
data[int_features_list] = data[int_features_list].apply(
    pd.to_numeric, errors='coerce', downcast='integer'
)
data[int_features_list].isnull().sum()

In [ ]:
# 2) Eliminar filas con NaN en esas columnas
data = data.dropna(subset=int_features_list)

# 3) Forzar a int (si quieres asegurarte que sean int "normales")
data[int_features_list] = data[int_features_list].astype(int)

In [ ]:
data_tracker.register_step(f"Cleaning int features with dropna", data)
data_tracker.to_dataframe()

### 1.3.5 Convertir a float (Real) las columnas que deberían ser flot y no lo son

In [ ]:
float_features_list = features[(features['Sub tipo'] == "Real") & (features['Sección'] != "Data Agregation")]['Característica'].unique()
float_features_list = list(float_features_list)

In [ ]:
# if ('Score CC' not in features_to_use) and ('Odds' in features_to_use):
#     float_features_list.append('Score CC')    
# display(float_features_list)

In [ ]:
# Convertir todo a floats (con NaN donde haya errores)
data[float_features_list] = data[float_features_list].apply(
    lambda col: pd.to_numeric(col, errors='coerce')
)

# Asegurar float64
data[float_features_list] = data[float_features_list].astype(float)

# Eliminar filas con problemas
data = data.dropna(subset=float_features_list)

print(data[float_features_list].info())
data_tracker.register_step(f"Cleaning na floats", data)
data_tracker.to_dataframe()

### 1.3.6 Identificar características cuantitativas y cualitativas

In [ ]:
for feature in quantitative_features : 
    if feature not in agregated_feautures:
        pkdp.categorical_print_values(feature, data)
        print("-----------------------")

In [ ]:
data.head(5)

In [ ]:
for feature in categorical_features :
    if feature not in agregated_feautures:
        pkdp.categorical_print_values(feature, data)
        print("-----------------------")

### 1.3.7 Limpiar datos numéricos

In [ ]:
numeric_features = features[(features['Numérico'] == 1) & (features['Sección'] != 'Data Agregation')]  
display(numeric_features)

In [ ]:
pkdp.print_sizes(data)
for index, row in numeric_features.iterrows():
    print(f"Column {row['Característica']} max value found: {data[row['Característica']].max()}")
    pkdp.clean_numeric_data(data, row['Característica'], row['Min'], row['Max'])
    print("=================================================")
pkdp.print_sizes(data)

In [ ]:
data_tracker.register_step(f"Cleaning numeric data min max", data)
data_tracker.to_dataframe()

In [ ]:
data.head()

# 2 Data agregation

## 2.1 Agregar $Deficit = Ingresos - Egresos$

In [ ]:
if 'Deficit' in quantitative_features:
    data['Deficit'] = data['Ingresos']-data['Egresos']
#quantitative_features.add('Deficit')

## 2.2 Agregar ingreso per capita

In [ ]:
if 'IngresoPercapita' in  quantitative_features:
    data['IngresoPercapita'] = data['Ingresos']/(data['Dependientes']+1)
#quantitative_features.add('IngresoPercapita')

## 2.2 Agregar la zona a partir del Codigo Postal

In [ ]:
#if 'TipoZona' in categorical_features:
#    CP_to_Zone = pd.read_excel("./CP_To_Zone.xlsx", index_col=0)
#    CP_Zone_set = set(CP_to_Zone.index)
#    data['TipoZona'] = data.apply(lambda x: CP_to_Zone.loc[x['CP']].Zona if x['CP'] in CP_Zone_set 
#                                  else "Unknown", axis=1)
#    data.head(5)

In [ ]:
cp_manager = pm_mxgeo.MX_CP_manager('./mlp/inputs/MX_CP_CoordZone.csv')

In [ ]:
data.columns

In [ ]:
data

In [ ]:
if 'TipoZona' in categorical_features:
    data['index'] = data.index
    print(data.head(5))
    data = data.merge(cp_manager.cp_base[['CP', 'TipoZona']], left_on='CP', right_on='CP', how='left').fillna("Fake")
    data = data.set_index(['index'])

In [ ]:
data

In [ ]:
data.columns

## 2.3 Agregar "Tiene aval" a partir de CP_Aval

In [ ]:
if 'Tiene aval' in categorical_features:
    data['Tiene aval'] = data.apply(lambda x: 0 if pd.isnull(x['CP Aval']) else 1, axis=1)

## 2.3 Agregar Latitud y Longitud a partir del CP

In [ ]:
data

In [ ]:
if features_to_use.intersection(pkdp.latitud_longitud_features_set):
    
    if 'CP_Latitud'in features_to_use: 
        print("Seting CP_Latitud")
        data = cp_manager.set_geographic_coordinate(data,'CP', 'CP_Latitud', 'Latitud')
        print(data.columns)
        print("-----------------------")
    
    if 'CP_Longitud' in features_to_use: 
        print("Seting CP_Longitud")
        data = cp_manager.set_geographic_coordinate(data,'CP', 'CP_Longitud', 'Longitud')
        print(data.columns)
        print("-----------------------")

    if 'CP_Aval_Latitud' in features_to_use: 
        data = cp_manager.set_geographic_coordinate(data,'CP Aval','CP_Aval_Latitud', 'Latitud')
        print(data.columns)
        print("-----------------------")

    if 'CP_Aval_Longitud' in features_to_use: 
        data = cp_manager.set_geographic_coordinate(data,'CP Aval','CP_Aval_Longitud', 'Longitud')
        print(data.columns)
        print("-----------------------")

In [ ]:
data

In [ ]:
data.info()

## 2.4 Agregar los odds de circulo de crédito

In [ ]:
if ('Odds' in quantitative_features):
    score_bins = [-1, 235, 406, 433, 451, 465, 477, 489, 499, 509, 518, 528, 538, 548, 559, 571, 584, 598, 614, 634, 664, 861]
    odds = [0, 0.90, 1.20, 1.30, 1.50, 1.60, 1.80, 1.90, 2.00, 2.10, 2.20, 2.60, 2.75, 2.90, 3.20, 3.50, 4.00, 4.60, 5.30, 6.10, 8.30]

    data['Odds'] = pd.cut(data['Score CC'], score_bins, labels=odds)
    data['Odds'] = data['Odds'].astype(float)
    print(data[['Score CC', 'Odds']])

In [ ]:
quantitative_features

In [ ]:
data.columns

## 2.5 Agregar el número de mes en que se solicita el crédito

In [ ]:
if ('Mes inicial' in quantitative_features):
    data['Fecha Inicial'] = pd.to_datetime(data['Fecha Inicial'], format='%d/%m/%Y')
    data['Mes inicial'] = data['Fecha Inicial'].dt.month

## 2.6 Transformación de columnas categóricas representadas como números

### 2.4.1 Mapear los estudios del cliente de un id entero a su significado

In [ ]:
if 'Estudios' in features_to_use:
    nivel_educativo = {
        0: 'Sin Estudios',
        1: 'Primaria',
        2: 'Secundaria',
        3: 'Preparatoria',
        4: 'Técnica',
        5: 'Licenciatura',
        6: 'Maestría',
        7: 'Carrera Trunca'
    }
    data['Estudios'] = data['Estudios'].map(nivel_educativo)

In [ ]:
data['Plazo'].head(5)

In [ ]:
if 'Plazo' in features_to_use:
    plazo_pago = {
        9: '9 Sem', 
        13: '13 Sem',
        16: '16 Sem',  
        17: '17 Sem',
        26: '26 Sem', 
        27: '27 Sem', 
    }
    data['Plazo'] = data['Plazo'].map(plazo_pago)

In [ ]:
data['Plazo'].head(5)

## 2.7 Usar solamente un numero N de dígitos en el código postal

In [ ]:
#if pkconfig.zip_digits <= 4:
#    if 'CP' in features_to_use: pk.transform_CP(data,'CP', pkconfig.zip_digits)
#    if 'CP Aval' in features_to_use: pk.transform_CP(data,'CP Aval', pkconfig.zip_digits)

## Borrar las columnas que no serán usados

In [ ]:
data = data.drop(labels=features_to_drop, axis=1)
data.head()

In [ ]:
pkdp.print_sizes(data)

In [ ]:
data.info()

In [ ]:
data.dtypes

In [ ]:
data.head()
#data.to_excel(pkconfig.data_file_name)
data.to_csv("./data/" + pkconfig.data_file_name)

In [ ]:
data_tracker.to_dataframe().to_csv("./data/" + pkconfig.conf_column + "_data_tracking.csv")

In [ ]:
features_to_drop


In [ ]:
time_now = datetime.now()

format_date_hour = "%Y-%m-%d %H:%M:%S"
date_and_hour = time_now.strftime(format_date_hour)

# Imprimir la fecha y hora
print("Ended at:", date_and_hour)

In [ ]:
data_tracker.to_dataframe()

In [ ]:
data.head(60)

In [ ]:
data = data.reset_index(drop=True)

In [ ]:
data.tail(60)

In [ ]:
# Imprimir la hora actual
print("Finalizado a las:", datetime.now().strftime("%H:%M:%S"))

# Hacer un sonido
if sys.platform.startswith("win"):
    import winsound
    winsound.Beep(1000, 500)  # Frecuencia de 1000 Hz, duración de 500 ms
    winsound.Beep(500, 500) 
    winsound.Beep(1000, 500) 
else:
    print('\a')  # Hace un beep en algunos sistemas UNIX
    
print('Configuration : ', pkconfig.conf_column)
pkdp.print_sizes(data)